# Stage-2 random baseline — T2 analysis (prompt 34 v2.1, scenario C)

The 48 uniform random lattice designs in `waves/random_C` are analyzed as **one
i.i.d. pool**. They are compared with the BO campaign (`bo_C_r1` + `bo_C_r2`)
on the same 32 Sobol starts. Everything here is pre-registered in prompt 34
§3: the statistic, the seeds, the verdict table, the robustness check and the
order of the secondaries. HV = `bogp.mobo.pareto.hypervolume_exact` on raw
`objectives.csv` units: shed MWh, curtailment MWh, cost USD. There is no g1
add-back before HV.

In [1]:
import os, sys, json, time, subprocess, itertools
import multiprocessing as mp
CAMP = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
os.environ.setdefault('BOGP_PATH', '/Users/yilu/Documents/GitHub/bo-gp-qparego')
sys.path.insert(0, os.environ['BOGP_PATH'])
sys.path.insert(0, CAMP + '/analysis/bo_stage2')
sys.path.insert(0, CAMP)
sys.path.insert(0, os.getcwd())
import acquire_next as an
from bogp.mobo.pareto import hypervolume_exact
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import make_batches as mb
import rb_hv as rb

led = an.load_ledger(CAMP)
head = subprocess.run(["git", "-C", os.environ["BOGP_PATH"], "rev-parse", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
assert head == led["bo_gp_sha"], (head, led["bo_gp_sha"])
print("bo-gp HEAD == ledger bo_gp_sha:", head)
M, T = an.M_COLS, an.SORTED_TIERS
assert M == rb.M_COLS and np.allclose(rb.FLOORS, an.FLOORS)
R_FROZEN = np.asarray(led["ref_point"], float)
FIG = os.path.join(os.getcwd(), "figs"); os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({"font.size": 11, "axes.labelweight": "bold", "axes.titleweight": "bold",
                     "axes.linewidth": 1.6, "xtick.major.width": 1.4, "ytick.major.width": 1.4,
                     "font.weight": "bold", "axes.spines.top": False, "axes.spines.right": False,
                     "savefig.dpi": 200, "savefig.bbox": "tight"})
C_START, C_BO, C_RND = "#8a8985", "#eb6834", "#2a78d6"
SCALE = np.array([1e-3, 1e-3, 1e-6])
LBL = ["Load shed [GWh]", "Curtailment [GWh]", "Cost less synthetic [$M]"]
NPROC = max(1, min(12, os.cpu_count() - 2))

bo-gp HEAD == ledger bo_gp_sha: 60e6328236e745b3d34dc8cd704bb72888ae5d15


## 0. Data (objectives merged onto design_matrix by `index`, as in `_load_wave_xy`)

In [2]:
frames = []
for w, src in [("stage2_C_n0", "start"), ("stage2_C_n0b", "start"), ("bo_C_r1", "bo"),
               ("bo_C_r2", "bo"), ("random_C", "random")]:
    dm = pd.read_csv(os.path.join(CAMP, "waves", w, "design_matrix.csv"))
    ob = pd.read_csv(os.path.join(CAMP, "waves", w, "objectives.csv"))
    d = dm.merge(ob[["index"] + M], on="index", validate="1:1")
    d["wave"], d["src"] = w, src
    frames.append(d)
D = pd.concat(frames, ignore_index=True)
D["label"] = (D["wave"].str.replace("stage2_C_", "").str.replace("bo_C_", "")
              .str.replace("random_C", "rnd") + "#" + D["index"].astype(str))
R = D[D.src == "random"].sort_values("index").reset_index(drop=True)
assert len(R) == 48 and list(R["index"]) == list(range(1, 49)) and np.isfinite(R[M].to_numpy()).all()
X40, Y40 = an.load_training(CAMP, 1)
X48, Y48 = an.load_training(CAMP, 2)
Y_S, Y_BO = Y48[:32], Y48[32:]
Y_R1, Y_R2 = Y48[32:40], Y48[40:48]
Y_POOL = rb.pool_matrix(R)                      # 48 x 3; arm never enters
assert np.allclose(np.sort(D[D.src != "random"][M].to_numpy(), 0), np.sort(Y48, 0))
# no random design repeats an evaluated one (lattice-index keys)
ev = {mb._lattice_index_key(x) for x in X48}
assert len(ev) == 48 and not ({mb._lattice_index_key(x) for x in R[[f"{t}_omega" for t in T]].to_numpy(float)} & ev)
print(D.groupby("src").size().to_dict(), "| random arms:", R.arm.value_counts().sort_index().to_dict())
Y_ALL96 = np.vstack([Y_S, Y_BO, Y_POOL])

{'bo': 16, 'random': 48, 'start': 32} | random arms: {1.0: 16, 2.0: 16, 3.0: 16}


## 1. R_union: frozen formula, computed once, before any HV

R_union = nadir + 0.1·span over all 96 evaluated designs (32 starts + 16 BO +
48 random). It is persisted to `ref_union.json`. If the file already exists,
it is read back and checked, never overwritten.

In [3]:
nadir, ideal = Y_ALL96.max(0), Y_ALL96.min(0)
R_UNION = nadir + 0.1 * (nadir - ideal)
ru_path = os.path.join(os.getcwd(), "ref_union.json")
if os.path.exists(ru_path):
    assert np.allclose(json.load(open(ru_path))["ref_union"], R_UNION, rtol=0, atol=0), "ref_union.json drifted"
else:
    json.dump({"ref_union": R_UNION.tolist(), "m_cols": M,
               "formula": "nadir + 0.1*span over the union of 96 evaluated designs (32 starts + 16 BO + 48 random)",
               "nadir": nadir.tolist(), "ideal": ideal.tolist(), "n_points": 96},
              open(ru_path, "w"), indent=2)
REF48 = np.asarray(json.load(open(os.path.join(CAMP, "analysis/bo_stage2/t2_summary.json")))
                   ["alt_ref_report_only"]["ref48"], float)
print(pd.DataFrame({"R_frozen": R_FROZEN, "R_union": R_UNION, "REF48 (BO_STAGE2)": REF48}, index=M).to_string())

                                   R_frozen       R_union  REF48 (BO_STAGE2)
load_shed_mwh                  1.510224e+04  3.209993e+04       3.209993e+04
true_curtailment_mwh           6.703497e+05  1.385278e+06       1.385278e+06
total_cost_less_synthetic_usd  6.520349e+08  6.843233e+08       6.843233e+08


## 2. Sanity gate: BO HV at n = 32/40/48 reproduces the ledger (relative error 1e-9)

In [4]:
for k, Yk in enumerate([Y_S, Y40, Y48]):
    a, b = hypervolume_exact(Yk, R_FROZEN), led["entries"][k]["hv"]
    print(f"n = {len(Yk)}: local {a:.10e}  ledger {b:.10e}  rel {abs(a-b)/b:.1e}")
    assert abs(a - b) / b < 1e-9
t0 = time.time()
for _ in range(1000): hypervolume_exact(Y48, R_FROZEN)
RATE_MS = (time.time() - t0)
print(f"HV rate: {RATE_MS:.3f} ms/call on the 48 evaluated points ({NPROC} worker processes)")

n = 32: local 6.6062715162e+17  ledger 6.6062715162e+17  rel 0.0e+00
n = 40: local 6.9692420940e+17  ledger 6.9692420940e+17  rel 0.0e+00
n = 48: local 7.0610491747e+17  ledger 7.0610491747e+17  rel 0.0e+00


HV rate: 0.448 ms/call on the 48 evaluated points (12 worker processes)


## 3. Primary statistic (§3.1)

- 100,000 random subsets of the pool, drawn without replacement (sampler seed
  20261012). The subset positions are paired across the two refs.
- Each subset is scored as HV(32 starts ∪ subset) / HV₀.
- p̂ = the fraction of subsets below BO; ties within 1e-12 relative count ½.
- Interval: 1,000 pool bootstraps × 2,000 subsets each, 5th–95th percentile.

In [5]:
N_SUB, N_BOOT, N_BSUB = 100_000, 1_000, 2_000
BO_N = {40: Y_R1, 48: Y_BO}
prim, HVS = {}, {}
t0 = time.time()
with mp.get_context("spawn").Pool(NPROC) as pool:
    for rname, ref in [("R_frozen", R_FROZEN), ("R_union", R_UNION)]:
        hv0 = hypervolume_exact(Y_S, ref)
        for n, size in [(40, 8), (48, 16)]:
            bo_hv = hypervolume_exact(np.vstack([Y_S, BO_N[n]]), ref)
            pos = rb.subset_positions(np.random.default_rng([rb.SUBSET_SEED, 0, size]), 48, size, N_SUB)
            chunks = np.array_split(pos, NPROC * 4)
            hvs = np.concatenate(pool.starmap(rb.subset_hvs, [(Y_S, Y_POOL, c, ref) for c in chunks]))
            p = rb.percentile_of(bo_hv, hvs)
            boots = np.array(pool.map(rb.boot_task, [(b, Y_S, Y_POOL, bo_hv, ref, size, N_BSUB)
                                                    for b in range(N_BOOT)], chunksize=8))
            lo, hi = np.percentile(boots, [5, 95])
            r = hvs / hv0
            HVS[(rname, n)] = r
            prim[(rname, n)] = dict(ref=rname, n=n, hv0=hv0, bo=bo_hv / hv0, p10=np.percentile(r, 10),
                                    p50=np.percentile(r, 50), p90=np.percentile(r, 90),
                                    bo_minus_median=bo_hv / hv0 - np.percentile(r, 50),
                                    p_hat=p, p_lo=lo, p_hi=hi, row=rb.verdict_row(p, lo, hi),
                                    boot_sd=boots.std())
            print(f"{rname} n={n}: p_hat={p:.4f} [{lo:.4f}, {hi:.4f}]  ({time.time()-t0:.0f} s)")
PRIM = pd.DataFrame(prim.values())
print(PRIM.drop(columns="hv0").round(4).to_string(index=False))
print("\nNote: the pool is finite (48 designs), so the subset spread slightly understates the variance of"
      "\nrandom sampling from the full lattice.")

R_frozen n=40: p_hat=0.9620 [0.8869, 1.0000]  (104 s)


R_frozen n=48: p_hat=0.9421 [0.8360, 1.0000]  (215 s)


R_union n=40: p_hat=0.9592 [0.8895, 1.0000]  (323 s)


R_union n=48: p_hat=1.0000 [1.0000, 1.0000]  (441 s)
     ref  n     bo    p10    p50    p90  bo_minus_median  p_hat   p_lo  p_hi   row  boot_sd
R_frozen 40 1.0549 1.0025 1.0211 1.0471           0.0339 0.9620 0.8869   1.0 above   0.0395
R_frozen 48 1.0688 1.0158 1.0421 1.0651           0.0268 0.9421 0.8360   1.0 above   0.0580
 R_union 40 1.0258 1.0019 1.0102 1.0216           0.0156 0.9592 0.8895   1.0 above   0.0414
 R_union 48 1.0460 1.0083 1.0204 1.0311           0.0255 1.0000 1.0000   1.0 above   0.0000

Note: the pool is finite (48 designs), so the subset spread slightly understates the variance of
random sampling from the full lattice.


In [6]:
# continuity: BO at BO_STAGE2's REF48 (report-only there)
h = [hypervolume_exact(Y, REF48) for Y in (Y_S, Y40, Y48)]
print("BO at REF48 (BO_STAGE2 wider box): HV/HV0 =", [round(x / h[0], 4) for x in h],
      "| gains % =", [round(100 * (h[i+1] - h[i]) / h[0], 2) for i in range(2)])
# provenance only: 3 arms vs BO, and the arbitrary rows 1-8 per arm
prov = []
for rname, ref in [("R_frozen", R_FROZEN), ("R_union", R_UNION)]:
    hv0 = hypervolume_exact(Y_S, ref)
    row = {"ref": rname, "BO n=48": hypervolume_exact(Y48, ref) / hv0, "BO n=40": hypervolume_exact(Y40, ref) / hv0}
    for k in (1, 2, 3):
        Ya = R[R.arm == k].sort_values("arm_order")[M].to_numpy(float)
        row[f"arm{k} n=48"] = hypervolume_exact(np.vstack([Y_S, Ya]), ref) / hv0
        row[f"arm{k} rows1-8 n=40"] = hypervolume_exact(np.vstack([Y_S, Ya[:8]]), ref) / hv0
    vals48 = [row["BO n=48"]] + [row[f"arm{k} n=48"] for k in (1, 2, 3)]
    row["BO rank among 4 (n=48)"] = int(1 + sum(v > row["BO n=48"] for v in vals48[1:]))
    row["whole 48-pool (n=80)"] = hypervolume_exact(np.vstack([Y_S, Y_POOL]), ref) / hv0
    prov.append(row)
PROV = pd.DataFrame(prov)
print(PROV.round(4).T.to_string())
print("Arm ranks are provenance only (under the null, BO ranks first 1 time in 4). The rows 1-8 values are"
      "\none arbitrary 8-subset each, illustrative only.")

BO at REF48 (BO_STAGE2 wider box): HV/HV0 = [1.0, 1.0258, 1.046] | gains % = [2.58, 2.02]
                               0        1
ref                     R_frozen  R_union
BO n=48                   1.0688    1.046
BO n=40                   1.0549   1.0258
arm1 n=48                 1.0581   1.0238
arm1 rows1-8 n=40         1.0456   1.0182
arm2 n=48                 1.0376   1.0202
arm2 rows1-8 n=40         1.0003   1.0031
arm3 n=48                 1.0293   1.0193
arm3 rows1-8 n=40         1.0069   1.0019
BO rank among 4 (n=48)         1        1
whole 48-pool (n=80)      1.0885   1.0446
Arm ranks are provenance only (under the null, BO ranks first 1 time in 4). The rows 1-8 values are
one arbitrary 8-subset each, illustrative only.


## 4. Verdict (§3.2 table, fixed before results)

In [7]:
WORD = {
    "above": "BO's HV is above most random 16-design batches at this budget (estimated percentile {p}, interval [{lo}, {hi}])",
    "matched": "random batches matched or exceeded BO's HV at this budget (percentile {p}, interval [{lo}, {hi}])",
    "comparable": "BO's HV is comparable to random sampling at this budget (percentile {p}, interval [{lo}, {hi}]); this budget cannot distinguish them",
}
fmt = lambda d: WORD[d["row"]].format(p=f"{d['p_hat']:.2f}", lo=f"{d['p_lo']:.2f}", hi=f"{d['p_hi']:.2f}")
f48, u48 = prim[("R_frozen", 48)], prim[("R_union", 48)]
if f48["row"] == u48["row"]:
    HEADLINE = (fmt(f48)[0].upper() + fmt(f48)[1:] + f" at R_frozen; at R_union: percentile {u48['p_hat']:.2f}, "
                f"interval [{u48['p_lo']:.2f}, {u48['p_hi']:.2f}].")
else:
    HEADLINE = f"ref-dependent: {fmt(f48)} [R_frozen] / {fmt(u48)} [R_union]."
MANDATORY = ("BO's total HV gain (6.9 % of HV₀) is below the campaign's per-round noise bar (8 %). "
             "No HV result in this experiment can show BO ahead by more than that bar.")
rows40 = {r: prim[(r, 40)]["row"] for r in ("R_frozen", "R_union")}
rows48 = {r: prim[(r, 48)]["row"] for r in ("R_frozen", "R_union")}
print("rows n=48:", rows48, "| rows n=40:", rows40)
print("\nHEADLINE:", HEADLINE, "\n\n", MANDATORY)

rows n=48: {'R_frozen': 'above', 'R_union': 'above'} | rows n=40: {'R_frozen': 'above', 'R_union': 'above'}

HEADLINE: BO's HV is above most random 16-design batches at this budget (estimated percentile 0.94, interval [0.84, 1.00]) at R_frozen; at R_union: percentile 1.00, interval [1.00, 1.00]. 

 BO's total HV gain (6.9 % of HV₀) is below the campaign's per-round noise bar (8 %). No HV result in this experiment can show BO ahead by more than that bar.


## 5. Robustness: paired floor perturbation (§3.3, report-only, no τ gate)

- 1,000 draws (seed 20261006).
- Each draw adds N(0, floor²) noise per objective to every evaluated point;
  the 32 starts get one common perturbation.
- p̂48 is recomputed with 2,000 subsets at the fixed R_frozen.
- Each draw is classified by its point row only.

In [8]:
t0 = time.time()
with mp.get_context("spawn").Pool(NPROC) as pool:
    pert = pool.map(rb.perturb_task, [(d, Y_S, Y_BO, Y_POOL, R_FROZEN, 2_000) for d in range(1_000)], chunksize=8)
p_pert = np.array([p for p, _ in pert]); bo_pert = np.array([b for _, b in pert])
det_point_row = rb.verdict_row(f48["p_hat"])
rows_pert = np.array([rb.verdict_row(p) for p in p_pert])
ROBUST_FRAC = float((rows_pert == det_point_row).mean())
print(f"({time.time()-t0:.0f} s) deterministic point row at R_frozen: {det_point_row}")
print("perturbed point rows:", pd.Series(rows_pert).value_counts().to_dict())
print(f"fraction matching: {ROBUST_FRAC:.3f} -> " + ("robust (>= 0.8)" if ROBUST_FRAC >= 0.8 else
      "NOT robust to floor-level output perturbation (< 0.8)"))
if ROBUST_FRAC < 0.8:
    HEADLINE += (f" Not robust to floor-level output perturbation (§3.3: {ROBUST_FRAC:.0%} of 1,000 perturbed draws"
                 f" keep the point row '{det_point_row}'; threshold 80%).")
print("\nFINAL HEADLINE:", HEADLINE)
print("p_hat48 under perturbation p5/p50/p95:", np.percentile(p_pert, [5, 50, 95]).round(3),
      "| BO HV/HV0 p5/p50/p95:", np.percentile(bo_pert, [5, 50, 95]).round(4))

(117 s) deterministic point row at R_frozen: above
perturbed point rows: {'comparable': 531, 'above': 467, 'matched': 2}
fraction matching: 0.467 -> NOT robust to floor-level output perturbation (< 0.8)

FINAL HEADLINE: BO's HV is above most random 16-design batches at this budget (estimated percentile 0.94, interval [0.84, 1.00]) at R_frozen; at R_union: percentile 1.00, interval [1.00, 1.00]. Not robust to floor-level output perturbation (§3.3: 47% of 1,000 perturbed draws keep the point row 'above'; threshold 80%).
p_hat48 under perturbation p5/p50/p95: [0.322 0.871 1.   ] | BO HV/HV0 p5/p50/p95: [1.0415 1.0905 1.248 ]


## 6. Pre-registered secondaries (§3.4)

*These are descriptive, with no multiplicity correction, and are not used in
the verdict.* They are printed in the fixed order 1–5.

### 6.1 n = 40 percentile

In [9]:
for r in ("R_frozen", "R_union"):
    d = prim[(r, 40)]
    print(f"{r}: p_hat40 = {d['p_hat']:.3f} [{d['p_lo']:.3f}, {d['p_hi']:.3f}] -> row '{d['row']}' "
          f"(n=48 row '{prim[(r, 48)]['row']}')")

R_frozen: p_hat40 = 0.962 [0.887, 1.000] -> row 'above' (n=48 row 'above')
R_union: p_hat40 = 0.959 [0.889, 1.000] -> row 'above' (n=48 row 'above')


### 6.2 S1: extremes attainment

BO's best-so-far at n = 48 is the best over 32 starts ∪ BO. The matched
random statistic is the best over 32 starts ∪ a random 16-subset, using the
same 100,000 subset positions as §3. The best within the subset alone is also
shown.

*Caveat: the cost extreme is guessable from physics (minimum build); S1
compares BO with uniform random, not with a domain heuristic.*

In [10]:
pos16 = rb.subset_positions(np.random.default_rng([rb.SUBSET_SEED, 0, 16]), 48, 16, N_SUB)
sub_best_only = Y_POOL[pos16].min(axis=1)                          # (N_SUB, 3)
sub_best = np.minimum(sub_best_only, Y_S.min(0))
bo_best = Y48.min(0)
assert np.allclose(bo_best, [0.0, 48444, 510.1e6], rtol=2e-3, atol=1.0), bo_best
lab48 = D[D.src != "random"].reset_index(drop=True)
s1 = []
for j, c in enumerate(M):
    pool_best, f = Y_POOL[:, j].min(), rb.FLOORS[j]
    s1.append({"objective": c, "BO best (32∪BO)": bo_best[j],
               "BO best from": ", ".join(lab48.loc[np.isclose(Y48[:, j], bo_best[j]), "label"]),
               "starts best": Y_S[:, j].min(),
               "P(subset best <= BO best - floor)": (sub_best[:, j] <= bo_best[j] - f).mean(),
               "P(subset best >= BO best + floor)": (sub_best[:, j] >= bo_best[j] + f).mean(),
               "subset best p10/p50/p90": np.percentile(sub_best[:, j], [10, 50, 90]).round(1).tolist(),
               "subset-only best p50": np.percentile(sub_best_only[:, j], 50),
               "48-pool best": pool_best, "pool best from": R.loc[Y_POOL[:, j].argmin(), "label"],
               "BO beats pool by > floor": bool(bo_best[j] < min(pool_best, Y_S[:, j].min()) - f),
               "BO beats pool-only by > floor": bool(bo_best[j] < pool_best - f)})
S1 = pd.DataFrame(s1)
S1_COUNT = int(S1["BO beats pool by > floor"].sum())
print(S1.T.to_string())
print(f"\nobjectives where BO's best beats the pool's best (32 ∪ pool) by more than the floor: {S1_COUNT} of 3"
      f"  (vs the 48 random designs alone: {int(S1['BO beats pool-only by > floor'].sum())} of 3)")
print("Caveat: the cost extreme is guessable from physics (minimum build); S1 compares BO with uniform random,"
      " not with a domain heuristic.")

                                                          0                            1                                        2
objective                                     load_shed_mwh         true_curtailment_mwh            total_cost_less_synthetic_usd
BO best (32∪BO)                                         0.0                 48444.091768                         510066070.036815
BO best from                       n0b#10, r1#5, r2#2, r2#6                         r1#5                                     r2#3
starts best                                             0.0                 76268.870635                         539251294.138745
P(subset best <= BO best - floor)                       0.0                          0.0                                      0.0
P(subset best >= BO best + floor)                       0.0                          1.0                                      1.0
subset best p10/p50/p90                     [0.0, 0.0, 0.0]  [61739.2, 76268.9, 76268.9]  

### 6.3 S2: grand front and recipe robustness

In [11]:
def nondominated(Y):
    return np.array([not np.any(np.all(Y <= y, axis=1) & np.any(Y < y, axis=1)) for y in Y])
D96 = pd.concat([lab48.assign(src=["start"] * 32 + ["bo"] * 16), R], ignore_index=True)
assert np.allclose(D96[M].to_numpy(float), Y_ALL96)
D96["front"] = nondominated(Y_ALL96)
D96["in_ref"] = np.all(Y_ALL96 < R_FROZEN, axis=1)
fr = D96.groupby("src").front.agg(["sum", "count"])
print("(i) grand front (96 points):", int(D96.front.sum()), "designs |", fr.to_dict("index"))
print(f"    BO on grand front: {fr.loc['bo','sum']}/16 = {fr.loc['bo','sum']/16:.3f};"
      f" random: {fr.loc['random','sum']}/48 = {fr.loc['random','sum']/48:.3f}")

N_S2 = 20_000
cnt_sub, cnt_bo, cnt_alone = np.empty(N_S2), np.empty(N_S2), np.empty(N_S2)
for i, p in enumerate(pos16[:N_S2]):
    nd = nondominated(np.vstack([Y_S, Y_BO, Y_POOL[p]]))
    cnt_bo[i], cnt_sub[i] = nd[32:48].sum(), nd[48:].sum()
    cnt_alone[i] = nondominated(np.vstack([Y_S, Y_POOL[p]]))[32:].sum()
bo_alone = int(nondominated(Y48)[32:].sum())
S2ii = {"E[subset count on front of 32∪BO∪subset]": cnt_sub.mean(),
        "p10/p90": np.percentile(cnt_sub, [10, 90]).tolist(),
        "E[BO count on the same front]": cnt_bo.mean(),
        "E[subset count on front of 32∪subset] (like-for-like with BO's 13/16)": cnt_alone.mean(),
        "p10/p90 (like-for-like)": np.percentile(cnt_alone, [10, 90]).tolist(),
        "BO count on front of 32∪BO": bo_alone}
print("(ii) subset-matched (first 20,000 of the §3 16-subsets):")
for k, v in S2ii.items(): print("    ", k, "=", np.round(v, 2))

(i) grand front (96 points): 32 designs | {'bo': {'sum': 13, 'count': 16}, 'random': {'sum': 12, 'count': 48}, 'start': {'sum': 7, 'count': 32}}
    BO on grand front: 13/16 = 0.812; random: 12/48 = 0.250


(ii) subset-matched (first 20,000 of the §3 16-subsets):
     E[subset count on front of 32∪BO∪subset] = 4.99
     p10/p90 = [3. 7.]
     E[BO count on the same front] = 13.0
     E[subset count on front of 32∪subset] (like-for-like with BO's 13/16) = 6.59
     p10/p90 (like-for-like) = [5. 8.]
     BO count on front of 32∪BO = 13


In [12]:
TIERS_D, _ = mb.stage2_tiers()
def knees_of(F, k=3):
    Yf = F[M].to_numpy(float); lo, hi = Yf.min(0), Yf.max(0)
    F = F.assign(dist_utopia=np.linalg.norm((Yf - lo) / (hi - lo), axis=1))
    return F.nsmallest(k, "dist_utopia")
rec = []
for norm, F in [("full front", D96[D96.front]), ("front clipped to R_frozen", D96[D96.front & D96.in_ref])]:
    for _, r in knees_of(F).iterrows():
        rc = mb.recipe_check({t: r[f"{t}_omega"] for t in T}, TIERS_D)
        rec.append({"normalization": norm, "front size": len(F), "knee": r.label, "src": r.src,
                    "dist": r.dist_utopia, **{t: r[f"{t}_omega"] for t in T}, "total_mw": rc["total_mw"],
                    "wind_share": rc["wind_share"], "in_R": rc["in_recipe"],
                    "shed": r[M[0]], "curt": r[M[1]], "cost_$M": r[M[2]] / 1e6})
KNEES = pd.DataFrame(rec)
print("(iii) recipe R =", mb.RANDOM_RECIPE)
print(KNEES.round(3).to_string(index=False))
ALL_KNEES_IN_R = bool(KNEES.in_R.all())
print("every knee of the 96-point front in R (both normalizations):", ALL_KNEES_IN_R)
for _, r in KNEES[~KNEES.in_R].iterrows():
    print(f"RECIPE COUNTER-EXAMPLE: {r.knee} ({r.normalization}) total {r.total_mw:.0f} MW, "
          f"wind share {r.wind_share:.2f}, nuclear {r.nuclear}, wind_122 {r.wind_122}")

(iii) recipe R = {'nuclear_omega': 0.05, 'total_gw': [1.6, 2.0], 'wind_303_317_share_min': 0.5, 'wind_122_omega_max': 0.2875}
            normalization  front size   knee    src  dist  nuclear    pv  tail  wind_122  wind_303  wind_317  total_mw  wind_share  in_R     shed       curt  cost_$M
               full front          32  rnd#2 random 0.403     0.05 0.762 0.169     0.169     0.406     0.881  1490.776       0.703 False 4663.407 295333.235  559.051
               full front          32  n0b#8  start 0.408     0.05 0.050 0.762     0.169     0.762     0.881  1699.241       0.794  True 1306.586 326405.377  563.115
               full front          32   r1#2     bo 0.416     0.05 0.288 0.881     0.288     0.762     0.644  1704.918       0.681  True  835.866 283798.849  568.229
front clipped to R_frozen          25   r2#1     bo 0.467     0.05 0.525 0.762     0.525     0.644     0.644  1824.796       0.581 False  785.282 221748.820  580.283
front clipped to R_frozen          25 rnd#43

### 6.4 Set coverage and per-arm front counts

In [13]:
SETS = {"BO": Y_BO, **{f"arm{k}": R[R.arm == k][M].to_numpy(float) for k in (1, 2, 3)}}
def coverage(A, B):   # fraction of B weakly dominated by some a in A
    return float(np.mean([np.any(np.all(A <= b, axis=1)) for b in B]))
COV = pd.DataFrame({a: {b: coverage(SETS[a], SETS[b]) for b in SETS} for a in SETS}).T
COV.index.name = "C(A, B): A ↓ / B →"
print(COV.round(3).to_string())
arm_front = {k: int(nondominated(np.vstack([Y_S, v]))[32:].sum()) for k, v in SETS.items()}
print("count of each set's 16 on the front of 32 ∪ set:", arm_front)

                     BO   arm1   arm2   arm3
C(A, B): A ↓ / B →                          
BO                  1.0  0.562  0.562  0.375
arm1                0.0  1.000  0.625  0.500
arm2                0.0  0.375  1.000  0.312
arm3                0.0  0.312  0.688  1.000
count of each set's 16 on the front of 32 ∪ set: {'BO': 13, 'arm1': 8, 'arm2': 6, 'arm3': 7}


### 6.5 Per-objective best-so-far at n = 32/40/48

In [14]:
pos8 = rb.subset_positions(np.random.default_rng([rb.SUBSET_SEED, 0, 8]), 48, 8, N_SUB)
bsf = {"BO": [Y_S.min(0), Y40.min(0), Y48.min(0)]}
for k in (1, 2, 3):
    Ya = R[R.arm == k].sort_values("arm_order")[M].to_numpy(float)
    bsf[f"arm{k} (rows in order)"] = [Y_S.min(0), np.vstack([Y_S, Ya[:8]]).min(0), np.vstack([Y_S, Ya]).min(0)]
bsf["random subset p50"] = [Y_S.min(0), np.median(np.minimum(Y_POOL[pos8].min(1), Y_S.min(0)), 0),
                            np.median(sub_best, 0)]
BSF = pd.DataFrame({(k, n): v[i] * np.array([1, 1, 1e-6]) for k, v in bsf.items() for i, n in enumerate([32, 40, 48])},
                   index=["shed MWh", "curt MWh", "cost $M"]).T
print(BSF.round(1).to_string())

                         shed MWh  curt MWh  cost $M
BO                   32       0.0   76268.9    539.3
                     40       0.0   48444.1    539.3
                     48       0.0   48444.1    510.1
arm1 (rows in order) 32       0.0   76268.9    539.3
                     40       0.0   76268.9    539.3
                     48       0.0   61739.2    539.3
arm2 (rows in order) 32       0.0   76268.9    539.3
                     40       0.0   76268.9    538.0
                     48       0.0   76268.9    538.0
arm3 (rows in order) 32       0.0   76268.9    539.3
                     40       0.0   76268.9    539.3
                     48       0.0   76268.9    536.0
random subset p50    32       0.0   76268.9    539.3
                     40       0.0   76268.9    539.3
                     48       0.0   76268.9    538.0


## 7. Figures

In [15]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=False)
for ax, rname in zip(axes, ["R_frozen", "R_union"]):
    ns = [32, 40, 48]
    p10 = [1.0] + [prim[(rname, n)]["p10"] for n in (40, 48)]
    p50 = [1.0] + [prim[(rname, n)]["p50"] for n in (40, 48)]
    p90 = [1.0] + [prim[(rname, n)]["p90"] for n in (40, 48)]
    bo = [1.0] + [prim[(rname, n)]["bo"] for n in (40, 48)]
    ax.fill_between(ns, p10, p90, color=C_RND, alpha=.22, lw=0, label="random subsets p10–p90")
    ax.plot(ns, p50, color=C_RND, lw=2, ls="--", marker="o", label="random subsets median")
    ax.plot(ns, bo, color=C_BO, lw=2.4, marker="s", ms=8, label="BO (this run)")
    for n in (40, 48):
        d = prim[(rname, n)]
        ax.annotate(f"p̂={d['p_hat']:.2f}", (n, d["bo"]), xytext=(6, 6), textcoords="offset points",
                    fontsize=9, fontweight="bold", color=C_BO)
    ax.set_xticks(ns); ax.set_xlabel("Evaluations n"); ax.set_ylabel("HV / HV₀")
    ax.set_title(f"HV trajectory at {rname}")
axes[0].legend(frameon=False, fontsize=9, loc="upper left")
fig.savefig(os.path.join(FIG, "fig1_hv_band.png")); fig.savefig(os.path.join(FIG, "fig1_hv_band.pdf"))

In [16]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for i, n in enumerate([48, 40]):
    for j, rname in enumerate(["R_frozen", "R_union"]):
        ax, d, r = axes[i, j], prim[(rname, n)], HVS[(rname, n)]
        ax.hist(r, bins=80, color=C_RND, alpha=.75)
        ax.axvline(d["bo"], color=C_BO, lw=2.4)
        ax.text(d["bo"], ax.get_ylim()[1] * .92, f" BO {d['bo']:.3f}\n p̂={d['p_hat']:.2f} [{d['p_lo']:.2f}, {d['p_hi']:.2f}]",
                color=C_BO, fontsize=9, fontweight="bold", va="top")
        ax.set_title(f"n = {n} ({n-32}-subsets), {rname}"); ax.set_xlabel("HV / HV₀ of 32 ∪ subset")
        ax.set_ylabel("subsets")
fig.tight_layout()
fig.savefig(os.path.join(FIG, "fig2_percentile_hist.png")); fig.savefig(os.path.join(FIG, "fig2_percentile_hist.pdf"))

In [17]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for j, ax in enumerate(axes):
    ax.hist(sub_best[:, j] * SCALE[j], bins=60, color=C_RND, alpha=.75, label="best of 32 ∪ random 16-subset")
    ax.axvline(bo_best[j] * SCALE[j], color=C_BO, lw=2.4, label="BO best (32 ∪ BO)")
    ax.axvspan((bo_best[j] - rb.FLOORS[j]) * SCALE[j], (bo_best[j] + rb.FLOORS[j]) * SCALE[j],
               color=C_BO, alpha=.12, lw=0, label="± floor")
    ax.set_xlabel(LBL[j]); ax.set_ylabel("subsets")
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle("S1: per-objective best-so-far at n = 48", fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG, "fig3_s1_extremes.png")); fig.savefig(os.path.join(FIG, "fig3_s1_extremes.pdf"))

In [18]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.3))
pairs = [(1, 2), (0, 2), (1, 0)]
kn = KNEES[KNEES.normalization == "full front"].knee
for ax, (a, b) in zip(axes, pairs):
    for src, c, mk, s in [("start", C_START, "o", 26), ("random", C_RND, "^", 30), ("bo", C_BO, "s", 34)]:
        d = D96[D96.src == src]
        ax.scatter(d[M[a]] * SCALE[a], d[M[b]] * SCALE[b], c=c, marker=mk, s=s, alpha=.35, lw=0)
        f = d[d.front]
        ax.scatter(f[M[a]] * SCALE[a], f[M[b]] * SCALE[b], c=c, marker=mk, s=s + 18, edgecolor="black", lw=.8,
                   label=f"{src} (on grand front: {len(f)})")
    k = D96[D96.label.isin(kn)]
    ax.scatter(k[M[a]] * SCALE[a], k[M[b]] * SCALE[b], marker="*", c="black", s=160, zorder=5, label="knees (full front)")
    ax.set_xlabel(LBL[a]); ax.set_ylabel(LBL[b])
axes[0].legend(frameon=False, fontsize=8, loc="upper right")
fig.suptitle("Grand front of 96 designs (outlined = non-dominated in 3-D)", fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(FIG, "fig4_grand_front.png")); fig.savefig(os.path.join(FIG, "fig4_grand_front.pdf"))

## 8. Summary numbers (consumed by RANDOM_BASELINE.md)

In [19]:
summary = {
    "headline": HEADLINE, "mandatory": MANDATORY, "rows48": rows48, "rows40": rows40,
    "primary": PRIM.to_dict("records"), "provenance": PROV.to_dict("records"),
    "ref_frozen": R_FROZEN.tolist(), "ref_union": R_UNION.tolist(), "ref48": REF48.tolist(),
    "robust": {"det_point_row": det_point_row, "frac_match": ROBUST_FRAC,
               "rows": pd.Series(rows_pert).value_counts().to_dict(),
               "p_pert_p5_p50_p95": np.percentile(p_pert, [5, 50, 95]).tolist()},
    "s1": S1.to_dict("records"), "s1_count": S1_COUNT,
    "s2_front": {"size": int(D96.front.sum()), "bo": int(fr.loc["bo", "sum"]), "random": int(fr.loc["random", "sum"]),
                 "start": int(fr.loc["start", "sum"])},
    "s2ii": S2ii, "knees": KNEES.to_dict("records"), "all_knees_in_R": ALL_KNEES_IN_R,
    "coverage": COV.to_dict(), "set_front_counts": arm_front,
    "bsf": {f"{k[0]}|{k[1]}": v.tolist() for k, v in BSF.iterrows()},
    "hv_rate_ms": RATE_MS, "nproc": NPROC,
    "seeds": {"subsets": "default_rng([20261012, 0, size])", "bootstrap": "default_rng([20261012, 1, size, b])",
              "perturb": "default_rng([20261006, d])"},
}
json.dump(summary, open(os.path.join(os.getcwd(), "t2_summary.json"), "w"), indent=2, default=float)
print(json.dumps({k: summary[k] for k in ("headline", "rows48", "rows40", "robust", "s1_count", "s2_front",
                                          "all_knees_in_R")}, indent=1, default=float))

{
 "headline": "BO's HV is above most random 16-design batches at this budget (estimated percentile 0.94, interval [0.84, 1.00]) at R_frozen; at R_union: percentile 1.00, interval [1.00, 1.00]. Not robust to floor-level output perturbation (\u00a73.3: 47% of 1,000 perturbed draws keep the point row 'above'; threshold 80%).",
 "rows48": {
  "R_frozen": "above",
  "R_union": "above"
 },
 "rows40": {
  "R_frozen": "above",
  "R_union": "above"
 },
 "robust": {
  "det_point_row": "above",
  "frac_match": 0.467,
  "rows": {
   "comparable": 531,
   "above": 467,
   "matched": 2
  },
  "p_pert_p5_p50_p95": [
   0.3215,
   0.871,
   1.0
  ]
 },
 "s1_count": 2,
 "s2_front": {
  "size": 32,
  "bo": 13,
  "random": 12,
  "start": 7
 },
 "all_knees_in_R": false
}
